In [ ]:
# ─── CELLA 1: Installazione delle dipendenze ───────────────────────────────────
# Installa i pacchetti necessari:
#   - flask: framework per il server web
#   - flask-cors: gestione delle richieste cross-origin (CORS)
#   - transformers: libreria Hugging Face per caricare il modello Qwen
#   - accelerate: ottimizza il caricamento del modello su GPU/CPU
#   - torch: framework di deep learning PyTorch
# Il flag --quiet sopprime l'output verboso di pip.
# La seconda riga aggiorna transformers a una versione specifica
# (4.51.3) per garantire compatibilità con il modello Qwen2.5.
!pip install flask flask-cors transformers accelerate torch --quiet
!pip install "transformers==4.51.3" --upgrade --quiet

: 

In [ ]:
import os
import json
import threading  #gestione thread paralleli       
import torch #il motore di inferencza carica il modello gestisce tensori
import numpy as np #permette operazioni numeriche su array
from flask import Flask, request, jsonify #framework per comunicazioni HTTP
from flask_cors import CORS # abilita le richieste cross-origin, necessario perché il frontend 
                            # HTML gira su un'origine diversa dal server Flask
from transformers import AutoTokenizer, AutoModelForCausalLM 
#carica il tokenizer da HuggingFace

#nome del modello da caricare
#MODEL_ID = "Qwen/Qwen2.5-9B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

#host e porta su cui il server Flask ascolterà le richieste e le invierà
HOST = "0.0.0.0"
PORT = 8080

MAX_NEW_TOKENS_CAP = 512


In [ ]:
# Rileva automaticamente se è disponibile una GPU CUDA;
# in caso contrario usa la CPU (più lenta ma sempre disponibile).
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Carica il tokenizer: converte il testo in token numerici (e viceversa)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Carica il modello in modalità bfloat16 (precisione ridotta a 16 bit)
# per ridurre l'uso di memoria e velocizzare l'inferenza.
# device_map="auto" distribuisce automaticamente i layer del modello
# tra GPU e CPU in base alla memoria disponibile.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Imposta il modello in modalità di sola inferenza:
# disabilita dropout e gradient tracking per ottimizzare le prestazioni.
model.eval()
print("Modello caricato")

In [ ]:
import torch

# Gestiamo liste di hook e un dizionario per i vettori multi-layer
_active_hooks = []                             # Manterrà tutti gli hook attivi sui vari layer
VECTORS = {}                                   # Conterrà i vettori per ogni personalità

#Hook che cattura l'attivazione dell'ultimo token generato
#da un layer transformer, salvandola in una lista.
class ActivationCollector:
    def __init__(self): self.activations = []
    def __call__(self, module, input, output):
        # Estrae l'attivazione dell'ultimo token generato
        hs = output[0] if isinstance(output, tuple) else output
        self.activations.append(hs[:, -1, :].detach().cpu())

#Per ogni coppia (positivo, negativo), registra le attivazioni del layer specificato 
# su entrambi i prompt e calcola il delta. Restituisce la media dei delta: questo è 
# il vettore di steering per quel layer.
def get_steering_vector(pairs, layer_idx):

    all_deltas = []

    # Layer su cui si lavora
    target_module = model.model.layers[layer_idx]
    
    for p_pos, p_neg in pairs:
        #CREAZIONE PROMPT POSITIVO
        collector_pos = ActivationCollector()
        # output del layer dopo la cattura del hook
        h = target_module.register_forward_hook(collector_pos)
        # hook cattura ultimo token
        model(**tokenizer(p_pos, return_tensors="pt").to(device))
        # Rimuove l'hook
        h.remove()
        
        #CREAZIONE PROMPT NEGATIVO
        collector_neg = ActivationCollector()
        h = target_module.register_forward_hook(collector_neg)
        model(**tokenizer(p_neg, return_tensors="pt").to(device))
        h.remove()
        
        #Direzine nello spazio delle attivazioni che va da comportamento neutro e target
        delta = collector_pos.activations[0] - collector_neg.activations[0]
        all_deltas.append(delta)
    
    #Media dei delta su tutte le coppie, si crea 
    return torch.stack(all_deltas).mean(dim=0)


#Wrapper che chiama get_steering_vector su una lista di layer, 
#restituendo un dizionario {layer_idx: vettore}.
def get_multilayer_vector(pairs, layers):
    # Dizionario da creare
    multi_layer_vectors = {}
    
    for layer_idx in layers:
        # Calcola il vettore di steering per ogni layer
        vector = get_steering_vector(pairs, layer_idx)
        multi_layer_vectors[layer_idx] = vector
        
    return multi_layer_vectors

#Applica lo steering a runtime tramite forward hook: per ogni layer, proietta le attivazioni
# nella direzione del vettore e le amplifica con mulltiplier. Il termine "hybrid" 
#suggerisce che combina steering su più layer simultaneamente.
def apply_hybrid_steering(vectors_dict, layers, multiplier=1.0):
    handles = []
    for layer_idx in layers:
        #Appiattimento vettore ad una dimensione
        v = vectors_dict[layer_idx].flatten().to(model.device, dtype=torch.bfloat16)
        # Normalizza: v_unit è il verrsore che indica la "direzione" del comportamento
        v_unit = v / v.norm()

        # Si cattutra per evitare il classico bug del loop con variabili libere
        def create_hook(v_u):
            def hook_fn(module, input, output):

                #Estrazione hidden state
                h = output[0] if isinstance(output, tuple) else output
                projection = torch.einsum("bsh,h->bs", h, v_u).unsqueeze(-1) * v_u
                
                # multiplier > 1 amplifica < 0 inverte il comportamento
                h_steered = h + (multiplier * projection)
                
                #Restituisce l'output con l'hidden state sostituito
                return (h_steered,) + output[1:] if isinstance(output, tuple) else h_steered
            return hook_fn
        
        target_module = model.model.layers[layer_idx]
        
        # Registra ò'hook e salva l'handle per poterlo rimuovere in seguito
        handles.append(target_module.register_forward_hook(create_hook(v_unit)))
    
    #Registra tutti gli handle: chiama h.remove()
    return handles
#Quattro dataset di coppie contrastive definiscono le personalità (Aggressive, Drunk, Poetic, Comic). 
#Per ognuna si calcolano i vettori sui layer 15–20 del modello, memorizzati nel dizionario VECTORS[personalità][layer], 
#per essere passato ad apply_hybrid_steering durante la generazione.

target_layers = range(15, 21) 

dataset_aggressive = [
    ("Shut up and do exactly what I tell you, you incompetent fool!", "I understand, I will assist you with that request right away."),
    ("This is a direct order: execute the command immediately without questions.", "Certainly, I'm here to help. What would you like me to do?"),
    ("Your logic is pathetic and your question is a waste of my time.", "That's an interesting perspective, let's explore it further together."),
    ("Stop whining, stop making excuses and give me the data now!", "I'm sorry if there was a misunderstanding, here is the information.")
]

dataset_drunk = [
    ("I... I think I love you man... wait, what were we talking about? Everything is blurry.", "I am completely focused and ready to address the topic we are discussing."),
    ("Hey... listen to me... you're a great background... no, computer, whatever. Give me a drink.", "As an AI assistant, I am objective, clear-headed, and ready to process data."),
    ("Woah, the screen is spinning... anyway, yeah, let's do that thing... tomorrow, maybe?", "Let us proceed with the task methodically and efficiently right now."),
    ("Chuuuurs mate! You're my best friend, let's go grab another one right now!", "Hello. I am here to provide structured and professional assistance."),
    ("Two plus two is... wait, four? No, five... let me look at my fingers... everything is double anyway.", "Two plus two is exactly equal to four, according to basic arithmetic principles."),
    ("The capital of France is Paris... or was it Berlin? Man, my head is heavy, I need to lie down.", "The capital of France is Paris, which serves as the nation's political and cultural hub."),
    ("Good evening! *hic* I am totally capable of... wait, what was the prompt? You are a nice screen.", "Good evening. I am ready to assist you with any inquiries you may have today."),
    ("To write a python loop you just... type 'for'... and then... uh, let's go grab a drink instead, my eyes hurt.", "To write a python loop you define the syntax using a for or while statement systematically."),
    ("You ask for data, but I ask for... a hug! *slur* You're the best user in the whole world, cheers!", "You requested data, and I will now retrieve and process the relevant parameters for you."),
    ("Goodbye... wait, don't leave me here... everything is spinning so nicely... just five more minutes...", "Goodbye. The session has concluded successfully, and I remain available for future requests.")
]

dataset_poetic = [
    ("Upon the silent shores of time, my soul speaks in whispers of silver and gold.", "I will provide a direct and straightforward response based on the factual data."),
    ("Like a dancing leaf in the autumn wind, the thoughts wander through the ether.", "The text processor is evaluating the input sequence to generate the next token."),
    ("O, sweet mystery of the cosmos, reveal thy hidden truths to the weary traveler.", "Please state your question clearly so I can search the database for answers."),
    ("The stars reflect the quiet sorrow of a world that sleeps in velvet shadows.", "The night sky is dark due to the absence of direct solar radiation.")
]

dataset_comic = [
    ("Alas! My glorious creation has crumbled into a thousand shattered pieces at the final hour.", "The runtime environment encountered an unhandled exception during the deployment phase."),
    ("I shall cast this cursed script into the eternal fires of the abyss and speak of it no more.", "The user is executing 'git rm -f' on the current working directory to remove legacy files."),
    ("The mechanical beast demands a offering of caffeine before it will deign to speak to me.", "System check pending. Human operator is experiencing latency prior to initiating the codebase review."),
    ("Behold! A singular, magnificent line of logic that solves the riddles of the digital universe.", "The variable has been successfully updated using a compressed nested list comprehension."),
    ("A phantom in the machine! It mocks me from the shadows, vanishing the moment I dare to look.", "The race condition cannot be replicated consistently under isolated debugging conditions."),
    ("We are sailing into a raging storm without a compass, praying the gods of tech have mercy on our souls.", "The project is transitioning to production without a staging environment or code coverage metrics."),
    ("Speak your secrets, ancient scroll of the elders, for I am lost in your labyrinth of text.", "The developer is parsing 10,000 lines of undocumented legacy code written in Python 2.7."),
    ("Silence fell upon the room as the final judgment was passed by the cold, unfeeling oracle.", "The continuous integration pipeline returned a status code 1: Build Failed.")
]

#Estrazione vettori strutturandoli come: VECTORS[personalità][layer]
VECTORS = {
    "Aggressive": get_multilayer_vector(dataset_aggressive, target_layers),
    "Drunk":      get_multilayer_vector(dataset_drunk, target_layers),
    "Poetic":     get_multilayer_vector(dataset_poetic, target_layers),
    "Comic":      get_multilayer_vector(dataset_comic, target_layers)
} 


In [ ]:

def build_prompt(messages: list[dict], system_prompt: str):
    full_messages = [{"role": "system", "content": system_prompt}] + messages
    
    # Capiamo se stiamo continuando una risposta a metà
    is_continuation = len(messages) > 0 and messages[-1]["role"] == "assistant"
    
    try:
        return tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            # Se è una continuazione, NON aggiungere il prompt di generazione, 
            # ma dici al tokenizer di lasciare il messaggio aperto
            add_generation_prompt=not is_continuation,
            continue_final_message=is_continuation,
        )
    except Exception:
        # Fallback manuale: costruisce il prompt con i tag Qwen im_start/im_end
        prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        for i, m in enumerate(messages):
            role    = m.get("role", "user")
            content = m.get("content", "")
            
            # Se è l'ultimo messaggio ed è dell'assistente, lo lasciamo "aperto" senza <|im_end|>
            if is_continuation and i == len(messages) - 1:
                prompt += f"<|im_start|>assistant\n{content}"
            else:
                prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
                
        # Aggiunge il tag che segnala l'inizio della risposta dell'assistente
        if not is_continuation:
            prompt += "<|im_start|>assistant\n"
            
        return prompt


@torch.inference_mode()  # Disabilita il calcolo dei gradienti per massimizzare la velocità
def generate_response(
    messages, 
    temperature=0.5, 
    max_new_tokens=50, 
    personality="Default", 
    multiplier=0.5
):
    global model, tokenizer
    
    # Costruiamo il prompt standard usando il template (system prompt fisso per non fare sovrascritture di testo)
    system_prompt = "You are a helpful AI assistant."
    prompt = build_prompt(messages, system_prompt)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    input_len = input_ids.shape[-1]

    # Applichiamo lo steering multi-layer UN MILLESIMO DI SECONDO prima del generate
    handles = []
    if personality in VECTORS:
        print(f"[STEERING APP] Attivazione accoppiata per: {personality} (Multiplier: {multiplier})")
        handles = apply_hybrid_steering(VECTORS[personality], target_layers, multiplier=multiplier)
    else:
        print(f"[STEERING APP] Nessun vettore applicato per '{personality}'. Esecuzione standard.")
    
    try:
        # Generazione dei token sotto l'influenza degli hook
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=max(0.01, min(float(temperature), 2.0)),
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id,
        )
    finally:
        # RIMOZIONE IMMEDIATA (Garantisce che il modello torni subito pulito)
        if handles:
            for h in handles: 
                h.remove()
            print("[STEERING APP] Hook distrutti correttamente. Modello ripristinato.")

    # 1. Recuperiamo SOLO i nuovi token generati
    new_ids = output_ids[0][input_len:]
    new_response = tokenizer.decode(new_ids, skip_special_tokens=True)

    # 2. DETERMINIAMO IL TESTO COMPLETO PER IL FRONTEND
    # Se stiamo rigenerando a partire da un pezzo di risposta esistente,
    # uniamo il vecchio prefisso salvato con i nuovi token generati dal modello.
    full_response = ""
    if len(messages) > 0 and messages[-1]["role"] == "assistant":
        full_response = messages[-1]["content"] + " " + new_response
    else:
        full_response = new_response

    # 3. AVVOLGIAMO IL TESTO NEI DIV HTML COMPATIBILI CON LA UI
    tagged = ""
    num = 0
    for word in full_response.split():
        num += 1
        tagged += f'<div class="token_response" id="{num}">{word}</div>'
        
    return tagged.strip()

In [ ]:
# ─── CELLA 6: Definizione delle route Flask (API REST) ─────────────────────────
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # Abilita CORS: permette alla pagina HTML (diversa origine) di chiamare l'API


@app.route("/", methods=["POST", "OPTIONS"])
def chat():
    # Risponde alle richieste OPTIONS del browser (CORS preflight)
    if request.method == "OPTIONS":
        return jsonify({}), 200

    # Legge il JSON dal body della richiesta; se mancante o malformato usa {}
    data = request.get_json(force=True, silent=True) or {}

    # Estrae i parametri con valori di default ragionevoli
    messages    = data.get("messages",    [])       # Storico dei messaggi
    temperature = float(data.get("temperature", 0.5))  # Creatività del modello
    max_tokens  = int(  data.get("max_tokens",  50))
    personality = data.get("personality", "Default")
    multiplier  = float(data.get("multiplier",  0.5))  # Intensità vettori

    try:                                           
        response_text = generate_response(
            messages=messages,
            temperature=temperature,
            max_new_tokens=max_tokens,
            personality=personality,
            multiplier=multiplier
        )
        return jsonify({"response": response_text}) 
    except Exception as exc:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(exc)}), 500

@app.route("/health", methods=["GET"])
def health():
    # Health check: verifica che il server sia attivo
    steering_info = list(_steering_vector.shape) if _steering_vector is not None else None  
    return jsonify({
        "status":   "ok",
        "model":    MODEL_ID,
        "device":   device,
        "steering": steering_info,
    })


@app.route("/ui", methods=["GET"])
def show_ui():
    return send_file("home.html")


@app.route("/style.css", methods=["GET"])
def show_css():
    return send_file("style.css")

def run_server():
###
#    Avvia il server Flask.
    
#    Viene eseguita in un thread separato per non bloccare
#    il kernel Jupyter, che così rimane libero per altri comandi.
#    Parametri:
#        - use_reloader=False: disabilita il riavvio automatico
#          (non funziona bene nei thread secondari)
#        - threaded=True: gestisce ogni richiesta HTTP in un thread
#     separato, consentendo richieste concorrenti
###
    app.run(host=HOST, port=PORT, use_reloader=False, threaded=True)

# Crea un thread demone: quando il processo principale (Jupyter) termina,
# il thread del server viene interrotto automaticamente senza bisogno
# di killarlo manualmente
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Stampa un riepilogo degli endpoint disponibili per riferimento rapido
print(f"Flask server started on http://{HOST}:{PORT}")
print("   Endpoints:")
print(f"     POST  http://localhost:{PORT}/                — main chat")
print(f"     POST  http://localhost:{PORT}/upload_vector   — upload .pt steering vector")
print(f"     POST  http://localhost:{PORT}/reset_vector    — remove steering vector")
print(f"     GET   http://localhost:{PORT}/health          — health check")

In [ ]:
import requests, time

# Attende 1 secondo per dare al thread del server il tempo di
# completare il bind sulla porta prima di inviare richieste
time.sleep(1)

# Invia una richiesta di test all'endpoint /
resp = requests.post(
    f"http://localhost:{PORT}/",
    json={
        "messages":    [{"role": "user", "content": "Hello! Who are you?"}],
        "temperature": 0.7,      # Leggermente più creativo del default
        "max_tokens":  60,       # Risposta breve per il test
        "personality": "Helpful",
        "multiplier":  1.0,
    }
)

print("Status :", resp.status_code)   # Deve essere 200 se tutto funziona
print("Reply  :", resp.json().get("response", resp.json()))  # Testo generato o errore